# scimap spatial analysis

Interactive scimap on the cleaned cells (09's `filtered_data.csv`, exported by `scripts/15_scimap_anndata.py`).

- Kernel: **Python (scimap)** (see `envs/scimap.yml` / CLAUDE.md).
- Coordinates are in µm (`X_centroid`, `Y_centroid`), so every `radius` below is in µm.
- `imageid` = section (`code`), `phenotype` = `cell_type2`, so scimap's default argument names work.
- `X` = asinh intensities (only needed for expression-based tools).

In [ ]:
import os
import numpy as np
import pandas as pd
import anndata as ad
import scimap as sm
import matplotlib.pyplot as plt
from scipy.stats import norm

PROJECT = os.environ.get("PCF_PROJECT", "/projects/b1169/boles/als_cns_pcf")
os.chdir(PROJECT)

REGION = "GM"        # "GM" or "WM"
RADIUS = 30          # um
N_PERM = 200         # interactive; use 1000 for final numbers
OUT = f"results/15_scimap/{REGION}/"
os.makedirs(OUT, exist_ok=True)

adata = ad.read_h5ad("data/15_scimap_anndata/cells.h5ad")
adata = adata[adata.obs["region"] == REGION].copy()
adata

## Composition per section

In [ ]:
sm.pl.stacked_barplot(adata, x_axis="imageid", y_axis="phenotype", method="percent")

## A. Nearest distance between cell types

`spatial_distance` gives every cell its distance to the nearest cell of each type (a cell is excluded as its own neighbour for its own type). The section-level table below (median per section × from type × to type) is what group comparisons should use.

In [ ]:
adata = sm.tl.spatial_distance(adata, phenotype="phenotype", imageid="imageid")
dist = adata.uns["spatial_distance"]
sm.pl.spatial_distance(adata, heatmap_summarize=True)

In [ ]:
d = dist.join(adata.obs[["imageid", "phenotype", "group", "sample_id"]])
nearest = (d.melt(id_vars=["imageid", "phenotype", "group", "sample_id"], var_name="to_type", value_name="dist_um")
             .groupby(["imageid", "group", "sample_id", "phenotype", "to_type"], observed=True)["dist_um"]
             .median().rename("median_dist_um").reset_index()
             .rename(columns={"imageid": "code", "phenotype": "from_type"}))
nearest.to_csv(OUT + "nearest_distance_medians.csv", index=False)
nearest.head()

## B. Neighbour interactions within a radius

`spatial_interaction` counts neighbours within `RADIUS` and compares them with `N_PERM` shuffles of the neighbours' labels, per section. In its output, each section's value is the neighbour frequency scaled to the row's maximum, signed by whether it is above (+) or below (−) the shuffled mean, and `pvalue_<section>` is a z-test p-value. That scaled value is not an observed/expected ratio, so the signed z recovered below (from the sign and p-value) is the per-section effect to compare between groups.

In [ ]:
adata = sm.tl.spatial_interaction(adata, phenotype="phenotype", imageid="imageid",
                                  method="radius", radius=RADIUS, permutation=N_PERM,
                                  label="spatial_interaction")
sm.pl.spatial_interaction(adata, spatial_interaction="spatial_interaction", p_val=0.05)

In [ ]:
si = adata.uns["spatial_interaction"]
codes = [c for c in si.columns if c not in ("phenotype", "neighbour_phenotype") and not c.startswith("pvalue_")]
rows = []
for c in codes:
    t = si[["phenotype", "neighbour_phenotype", c, "pvalue_" + c]].copy()
    t.columns = ["from_type", "to_type", "score", "p"]
    t["code"] = c
    rows.append(t)
inter = pd.concat(rows, ignore_index=True)
inter["z"] = np.sign(inter["score"]) * norm.isf(inter["p"].clip(lower=1e-300) / 2)
meta = adata.obs[["imageid", "group", "sample_id"]].drop_duplicates().rename(columns={"imageid": "code"})
inter = inter.merge(meta, on="code")
inter.to_csv(OUT + f"interaction_r{RADIUS}.csv", index=False)
inter.head()

## Neighbourhoods (exploratory)

Cell-type composition within `RADIUS` of each cell, then k-means into `k` neighbourhoods (the Schürch 2020 / scimap approach).

In [ ]:
adata = sm.tl.spatial_count(adata, phenotype="phenotype", imageid="imageid", method="radius", radius=RADIUS,
                            label="spatial_count")
adata = sm.tl.spatial_cluster(adata, df_name="spatial_count", method="kmeans", k=10, label="neighborhood")
sm.pl.stacked_barplot(adata, x_axis="neighborhood", y_axis="phenotype", method="percent")

In [ ]:
sm.pl.stacked_barplot(adata, x_axis="imageid", y_axis="neighborhood", method="percent")

In [ ]:
adata.write_h5ad(OUT + "cells_scimap.h5ad")